In [1]:
# =============================================================
# PERSON 2 — DEEPER CNN + HYPERPARAMETER TUNING
# =============================================================

# ── WHAT EACH LAYER DOES ─────────────────────────────────────
#
# Conv2D:
#   This layer looks at small sections of the image at a time
#   and tries to find patterns like edges and shapes. The more
#   conv layers you have the better the model gets at recognizing
#   complex things like finger positions.
#
# MaxPooling:
#   After each conv layer we shrink the image down to keep only
#   the most important information. This makes the model faster
#   and stops it from focusing on unnecessary details.
#
# Dropout:
#   During training this randomly turns off some neurons so the
#   model cant just memorize the training images. It forces it to
#   actually learn how to recognize the gestures properly.
#
# Dense:
#   This is the final decision maker. It takes everything the
#   conv layers found and uses it to decide which of the 29 ASL
#   letters the image is showing.
#
# ── MY ARCHITECTURE ──────────────────────────────────────────
#
#   I used 4 convolutional layers which is deeper than Person 1s
#   model. More layers means the model can pick up on more complex
#   patterns in the hand shapes.
#
#   Layer 1: Conv2D (32 filters)  + MaxPooling
#   Layer 2: Conv2D (64 filters)  + MaxPooling
#   Layer 3: Conv2D (128 filters) + MaxPooling
#   Layer 4: Conv2D (128 filters)
#   Flatten
#   Dense (256 neurons)
#   Dropout (0.5)
#   Output: 29 neurons with Softmax (one per ASL class)
#
# ── TUNING EXPERIMENT ────────────────────────────────────────
#
#   I trained 3 versions changing the dropout rate and dense
#   layer size each time to see what worked best:
#
#   Version | Dropout | Dense Units | Val Accuracy
#   --------|---------|-------------|-------------
#   V1      | 0.3     | 128         | 99.71%
#   V2      | 0.5     | 128         | 99.17%
#   V3      | 0.5     | 256         | 99.90% ← best
#
#   V3 gave the best result. The higher dropout stopped the model
#   from overfitting and the bigger dense layer of 256 neurons
#   gave it more room to learn the differences between all 29
#   hand gestures. So I saved V3 as the final model.
#
# =============================================================



import numpy as np
import torch
import torch.nn as nn

# Load data
X_train = np.load('X_train.npy')
X_test  = np.load('X_test.npy')
y_train = np.load('y_train.npy')
y_test  = np.load('y_test.npy')

print("X_train shape:", X_train.shape)

# Convertint to tensors
X_train = torch.tensor(X_train, dtype=torch.float32).permute(0, 3, 1, 2) #.permute to rearrange data 
X_test  = torch.tensor(X_test,  dtype=torch.float32).permute(0, 3, 1, 2)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test,  dtype=torch.long)

print("Tensors ready!")
print("X_train tensor shape:", X_train.shape)

# Cnn starts here 
class DeepCNN(nn.Module):
    def __init__(self, dropout_rate, dense_units):
        super(DeepCNN, self).__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 128, 3, padding=1),nn.ReLU()
        )
        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, dense_units),
            nn.ReLU(),
            nn.Dropout(dropout_rate), # to prevent overfiiting 
            nn.Linear(dense_units, 29)
        )

    def forward(self, x):
        x = self.conv_block(x)
        x = self.fc_block(x)
        return x

print("Model class defined!")

from torch.utils.data import TensorDataset, DataLoader


train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

# traininggg :> 
def train_model(dropout_rate, dense_units, version_name):
    print(f"\nTraining {version_name} — dropout={dropout_rate}, dense={dense_units}")
    
    model     = DeepCNN(dropout_rate, dense_units)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(15):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

       
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                preds = model(X_batch).argmax(dim=1)
                correct += (preds == y_batch).sum().item()
                total   += y_batch.size(0)
        
        acc = correct / total * 100
        print(f"  Epoch {epoch+1}/15 — Val Accuracy: {acc:.2f}%")

    torch.save(model.state_dict(), f'{version_name}.pth')
    print(f"  Saved as {version_name}.pth")
    return acc


v1_acc = train_model(dropout_rate=0.3, dense_units=128, version_name="model_v1")
v2_acc = train_model(dropout_rate=0.5, dense_units=128, version_name="model_v2")
v3_acc = train_model(dropout_rate=0.5, dense_units=256, version_name="model_v3")

print("\n===== RESULTS =====")
print(f"V1 (dropout=0.3, dense=128): {v1_acc:.2f}%")
print(f"V2 (dropout=0.5, dense=128): {v2_acc:.2f}%")
print(f"V3 (dropout=0.5, dense=256): {v3_acc:.2f}%")

X_train shape: (62400, 64, 64, 1)
Tensors ready!
X_train tensor shape: torch.Size([62400, 1, 64, 64])
Model class defined!

Training model_v1 — dropout=0.3, dense=128
  Epoch 1/15 — Val Accuracy: 84.84%
  Epoch 2/15 — Val Accuracy: 93.55%
  Epoch 3/15 — Val Accuracy: 97.24%
  Epoch 4/15 — Val Accuracy: 98.01%
  Epoch 5/15 — Val Accuracy: 98.47%
  Epoch 6/15 — Val Accuracy: 98.66%
  Epoch 7/15 — Val Accuracy: 99.12%
  Epoch 8/15 — Val Accuracy: 97.69%
  Epoch 9/15 — Val Accuracy: 99.20%
  Epoch 10/15 — Val Accuracy: 99.30%
  Epoch 11/15 — Val Accuracy: 99.66%
  Epoch 12/15 — Val Accuracy: 99.35%
  Epoch 13/15 — Val Accuracy: 99.71%
  Epoch 14/15 — Val Accuracy: 99.80%
  Epoch 15/15 — Val Accuracy: 99.71%
  Saved as model_v1.pth

Training model_v2 — dropout=0.5, dense=128
  Epoch 1/15 — Val Accuracy: 73.18%
  Epoch 2/15 — Val Accuracy: 85.67%
  Epoch 3/15 — Val Accuracy: 92.94%
  Epoch 4/15 — Val Accuracy: 95.29%
  Epoch 5/15 — Val Accuracy: 95.53%
  Epoch 6/15 — Val Accuracy: 98.09%
  E